<a href="https://colab.research.google.com/github/SoulDataLab/CODSOFT_TASKSNO/blob/main/CODSOFT_Task1_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

file_path = "marketing_campaign.csv"
df = pd.read_csv(file_path, sep="\t")

print("Dataset loaded successfully into DataFrame.")

In [ ]:
print(" FIRST 5 RECORDS OF THE DATASET ")
display(df.head(5))

print("\n LAST 5 RECORDS OF THE DATASET ")
display(df.tail(5))

In [ ]:
print(" DATASET DIMENSIONS OVERVIEW ")
print(f"Dataset Shape (Rows, Columns): {df.shape}")
print(f"Total Number of Rows         : {df.shape[0]}")
print(f"Total Number of Columns      : {df.shape[1]}")
print(f"Total Elements in Dataset    : {df.size}")

print("\n=== LIST OF ALL COLUMN NAMES ===")
for idx, col in enumerate(df.columns, start=1):
    print(f"{idx:2d}. {col}")

In [ ]:
print(" DATA TYPES SUMMARY ")
print(df.dtypes.value_counts())

print("\n DETAILED STRUCTURE AND MEMORY INFO ")
df.info()

In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    'Total_Missing_Values': missing_count,
    'Missing_Percentage (%)': missing_percentage.round(2)
})

print(" FULL MISSING VALUES REPORT ")
display(missing_report)

print("\n COLUMNS WITH MISSING VALUES ONLY ")
display(missing_report[missing_report['Total_Missing_Values'] > 0])

In [ ]:
exact_duplicates = df.duplicated().sum()
feature_columns = [col for col in df.columns if col != 'ID']
profile_duplicates = df.duplicated(subset=feature_columns).sum()

print(" DUPLICATE RECORDS SUMMARY ")
print(f"Exact Full-Row Duplicates (Including ID)     : {exact_duplicates}")
print(f"Duplicate Customer Profiles (Excluding ID)   : {profile_duplicates}")

print("\n SAMPLE OF DUPLICATE CUSTOMER PROFILES ")
display(df[df.duplicated(subset=feature_columns, keep=False)].sort_values(by=['Year_Birth', 'Income']).head(6))

In [ ]:
print(" UNIQUE VALUES IN EDUCATION COLUMN ")
print(df['Education'].value_counts())

print("\n UNIQUE VALUES IN MARITAL_STATUS COLUMN ")
print(df['Marital_Status'].value_counts())

In [ ]:
print(" YEAR_BIRTH RANGE CHECK ")
print(f"Minimum Birth Year : {df['Year_Birth'].min()}")
print(f"Maximum Birth Year : {df['Year_Birth'].max()}")

print("\n CUSTOMERS WITH BIRTH YEAR BEFORE 1940 (INCONSISTENT ENTRIES) ")
display(df[df['Year_Birth'] < 1940][['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income']])

constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"\n CONSTANT COLUMNS (NO VARIATION IN DATA): {constant_cols} ")
for col in constant_cols:
    print(f"{col} unique value: {df[col].unique()}")

In [ ]:
# Fill missing values in Income column with the median income
median_income = df['Income'].median()
df['Income'] = df['Income'].fillna(median_income)

# Clean Marital_Status by mapping rare or inconsistent categories
df['Marital_Status'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
})

# Remove unrealistic birth years (customers born before year 1940)
df = df[df['Year_Birth'] >= 1940].reset_index(drop=True)

print("Missing values after cleaning:", df.isnull().sum().sum())
print("Data shape after filtering outliers:", df.shape)

In [ ]:
# Remove duplicate customer profiles while keeping the first occurrence (ignoring ID column)
feature_columns = [col for col in df.columns if col != 'ID']
initial_count = len(df)
df = df.drop_duplicates(subset=feature_columns).reset_index(drop=True)
final_count = len(df)

print(f"Total duplicate rows removed: {initial_count - final_count}")
print(f"Final dataset shape after removing duplicates: {df.shape}")

In [ ]:
# Drop constant columns that carry no analytical value
columns_to_drop = [col for col in ['Z_CostContact', 'Z_Revenue'] if col in df.columns]
df = df.drop(columns=columns_to_drop)

# Save the cleaned dataset as a new CSV file
output_file_name = "cleaned_marketing_campaign.csv"
df.to_csv(output_file_name, index=False)

print(f"Cleaned dataset successfully saved as '{output_file_name}'.")
print("=== FINAL CLEANED DATASET PREVIEW ===")
display(df.head(3))